# The Machine Learning Workflow
Every step of a real machine learning project, from question to working model. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. Get the data
A café wants to predict how many cups of coffee it will sell each day, so it orders the right amount of milk and beans. The **features** are what we know in advance (day, temperature, rain, holiday). The **target** is what we want to predict (cups sold).

### 400 days of café sales
- `cafe_sales.csv`: 400 days: day of the week, temperature in degrees Celsius, rain and holiday (1 = yes), and cups sold
- `df.shape`: 400 rows (days) and 5 columns
- `df.head()`: the first five days

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "machine-learning-workflow/cafe_sales.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

## 2. Explore the data
Before building anything, look at the data: summaries, problems, and patterns.

### A quick summary
- `df.describe()`: count, mean, smallest, largest and more, for each number column
- `max temperature 999`: impossible: a typing mistake we must fix
- `count 390`: not 400: some temperatures are missing

In [ ]:
df.describe().round(1)

### Missing values
- `df.isna()`: True wherever a value is missing
- `.sum()`: counts them in each column: 10 temperatures

In [ ]:
df.isna().sum()

### Temperature against cups sold
- `df[df["temperature"] < 60]`: leave out the impossible 999 just for this plot
- `plt.scatter`: one dot per day
- `the pattern`: the colder the day, the more coffee sold

In [ ]:
import matplotlib.pyplot as plt

ok = df[df["temperature"] < 60]
plt.scatter(ok["temperature"], ok["cups_sold"], alpha=0.5)
plt.xlabel("temperature (C)")
plt.ylabel("cups sold")
plt.title("Colder days sell more coffee")
plt.show()

## 3. Clean and prepare
Fix mistakes, and turn everything into numbers, because models only understand numbers.

### Fix the impossible value
- `df["temperature"] > 60`: True for the impossible row
- `~`: means NOT: keep every row except that one (missing temperatures are kept for now)
- `399 rows`: one row removed

In [ ]:
df = df[~(df["temperature"] > 60)]
print(df.shape)

### Turn the day into a number
- `.isin(["Sat", "Sun"])`: True for Saturday and Sunday
- `.astype(int)`: True becomes 1, False becomes 0
- `X and y`: by tradition, X holds the features and y holds the target

In [ ]:
df["weekend"] = df["day"].isin(["Sat", "Sun"]).astype(int)
features = ["temperature", "rain", "holiday", "weekend"]
X = df[features]
y = df["cups_sold"]
print(X.head())

## 4. Split into training and test sets
Keep some days aside as a **test set**. The model never sees them while learning, so they show how it does on new days.

### Train / test split
- `test_size=0.2`: keep 20% of the days for testing
- `random_state=42`: shuffle the same way every time, so results repeat
- `four results`: features and target, for training and for testing

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(len(X_train), "training days")
print(len(X_test), "test days")

### Fill missing temperatures (after the split)
- `X_train[...].median()`: the middle temperature, worked out from the training days only
- `fillna`: put that value wherever a temperature is missing, in both sets
- `why training only`: using the test days here would leak information from the test (data leakage)

In [ ]:
median_temp = X_train["temperature"].median()
X_train = X_train.fillna({"temperature": median_temp})
X_test = X_test.fillna({"temperature": median_temp})
print("filled with", median_temp)

## 5. A baseline
The simplest possible guess. Any real model must beat it.

### Always guess the average
- `y_train.mean()`: the average cups sold on the training days
- `[guess] * len(y_test)`: the same guess for every test day
- `mean_absolute_error`: on average, how many cups off the guesses are

In [ ]:
from sklearn.metrics import mean_absolute_error

guess = y_train.mean()
base_pred = [guess] * len(y_test)
print("guess:", round(guess))
print("error:", round(mean_absolute_error(y_test, base_pred), 1))

## 6. Train a model
A **decision tree** learns a set of yes/no questions from the training data, like a game of twenty questions.

### Train a decision tree
- `DecisionTreeRegressor`: a decision tree that predicts a number (regression)
- `max_depth=3`: ask at most 3 questions in a row
- `model.fit(X, y)`: learn from the training days: every scikit-learn model uses fit

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(max_depth=3, random_state=42)
model.fit(X_train, y_train)
print("trained")

### What did it learn?
- `export_text`: writes the tree's questions as text
- `weekend <= 0.5`: is it a weekday? (weekend is 0 or 1)
- `[:8]`: just the first 8 lines, so it fits

In [ ]:
from sklearn.tree import export_text

rules = export_text(model, feature_names=features,
                    decimals=1)
print("\n".join(rules.split("\n")[:8]))

## 7. Evaluate on the test set
Now the honest test: predict the days the model has never seen.

### Test error
- `model.predict`: every scikit-learn model also uses predict
- `error`: about 11 cups off on average, half the baseline's error (about 22)
- `first 5`: predictions next to the real numbers

In [ ]:
pred = model.predict(X_test)
error = mean_absolute_error(y_test, pred)
print("model error:", round(error, 1))
print("first 5 predictions:", pred[:5].round())
print("first 5 actual:     ", y_test.values[:5])

## 8. Improve
Try a few settings. Too simple misses patterns; too complex memorises the training days.

### Trying different depths
- `for depth in [...]`: train one tree for each setting
- `train error`: keeps falling as the tree grows: it memorises
- `test error`: falls, then rises again after depth 5: memorising doesn't help on new days (overfitting)

*Choosing settings by looking at the test set again and again slowly leaks it; real projects use a separate validation set for this.*

In [ ]:
for depth in [1, 2, 3, 5, 10, 20]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    tr = mean_absolute_error(y_train, m.predict(X_train))
    te = mean_absolute_error(y_test, m.predict(X_test))
    print(f"depth {depth:>2}:  train {tr:5.1f}   test {te:5.1f}")

## 9. Use the model
Predict tomorrow, and save the model so the café's app can use it.

### Predict tomorrow
- `max_depth=5`: the best setting we found: train the final model with it
- `tomorrow`: a cold, rainy Saturday, with the same four features, in the same order
- `model.predict(...)[0]`: the prediction for that one day

In [ ]:
model = DecisionTreeRegressor(max_depth=5, random_state=42)
model.fit(X_train, y_train)

tomorrow = pd.DataFrame([{"temperature": 12, "rain": 1,
                          "holiday": 0, "weekend": 1}])
print("cups to prepare:", round(model.predict(tomorrow)[0]))

### Save and load the model
- `joblib.dump`: saves the trained model to a file
- `joblib.load`: loads it back later, or in another program, with no retraining

In [ ]:
import joblib

joblib.dump(model, "cafe_model.joblib")
loaded = joblib.load("cafe_model.joblib")
print(round(loaded.predict(tomorrow)[0]))

## Try it yourself
1. Predict sales for a hot, dry Monday (28 degrees, no rain, no holiday).
2. Change `max_depth` to 2 and to 8. What happens to the test error?
3. Change `test_size` to 0.3 and run everything again.